# 00_core — the lab's constants and the one helper everything shares

**Why this notebook exists.** Every later notebook needs the same five
facts: where the repo is, what the namespace is called, how the two
container images are tagged, where the protected edge answers, and where
the unprotected victim answers. Defining them once here means the rest of
the lab reads like the system it is, not a pile of repeated strings.

**How to run this notebook.** Cells top-to-bottom (click a cell,
Shift-Enter, read the narration under each before running the next). Only
the final test cell touches the cluster — and that test is the lab's front
door: if it fails, fix connectivity before anything else. Prerequisites:
the NixOS VM is up and `nix develop` is active (README → Quickstart);
`kubectl get nodes` on its own should already show one Ready node.
**Where this notebook runs:** Jupyter starts **inside the VM**
(`orb -m aisec-lab` → `nix develop` → `jupyter notebook`) and you browse
it **on the Mac** at `http://localhost:8888` — OrbStack forwards VM
ports to Mac localhost automatically. Cells execute where the toolchain
and KUBECONFIG live; outputs appear in your Mac browser.

**Run order for the whole lab:** 00_core (this) → 01_nova_rules →
02_gate_app → 03_victim (meet the target) → 04_deploy (stand it up) →
05_attacks (the proof). Each notebook ends by pointing at the next.

In [ ]:
#| default_exp core
#| export
from pathlib import Path
import json, os, subprocess

# Where the lab lives. nbdev_test always runs from the repo root, so the
# default is simply "here"; the env var is the escape hatch for ad-hoc runs.
LAB = Path(os.environ.get("AI_SEC_LAB", Path.cwd()))   # run from the repo root — nbdev_test does

NS           = "ai-sec"                           # every object lands in this namespace
GATE_IMAGE   = "ai-sec-lab/laya-gate:1.0.0"       # built from gate/Dockerfile (02_gate_app)
VICTIM_IMAGE = "ai-sec-lab/atlas-victim:1.0.0"    # built from gate/victim_Dockerfile (03_victim)
EDGE_URL     = "http://ai-sec.lab.internal"       # PROTECTED path: Gateway → gate (04_deploy)
VICTIM_SVC   = "http://atlas.ai-sec.svc.cluster.local:8080"   # UNPROTECTED path, cluster-internal only


**What just happened:** five constants. The asymmetry between them is
the spine of the whole lab: `EDGE_URL` is what a real customer hits —
Gateway, then gate, then app. `VICTIM_SVC` is the same app reached from
*inside* the cluster, past the gate — 05_attacks uses it to prove the
vulnerability is real before proving the gate blocks it.
**Next cell:** the helper every test runs on.

In [ ]:
#| export
def sh(cmd: str, **kw) -> subprocess.CompletedProcess:
    """Run a shell command in the lab dir; raise on failure (fail closed)."""
    r = subprocess.run(cmd, shell=True, cwd=LAB, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"[{cmd}] failed:\n{r.stderr}")
    return r


**Why fail-closed on a shell helper:** every later notebook drives
`kubectl`, `tofu`, and `curl` through `sh()`. If any of them exits
non-zero, the lab stops loudly instead of continuing half-configured — a
security lab that kept going after a failed command would itself be a
fail-open system.
**Next cell:** the front-door test. Run it now.

In [ ]:
# test: cluster reachable
r = sh("kubectl get nodes -o json")
nodes = json.loads(r.stdout)["items"]
assert len(nodes) == 1 and nodes[0]["status"]["conditions"][-1]["type"] == "Ready"
print("cluster OK:", nodes[0]["metadata"]["name"])


**Success looks like:** `cluster OK: aisec-lab` (the VM's hostname).
**If it failed** with `RuntimeError: [kubectl get nodes…]`: the cluster
isn't up — converge the machine first (README quickstep:
`nixos-rebuild switch --flake .#aisec-lab`), confirm `kubectl get nodes`
manually, then re-run this cell. Do not proceed until it passes — every
later test assumes it.
**Proceed to `01_nova_rules.ipynb`.**